In [ ]:
!pip uninstall cupy-cuda13x

# 1. Install core PySCF (compiled for your CPU host)
!pip install --prefer-binary pyscf

# 2. Install the GPU4PySCF extension pre-compiled for CUDA 12
!pip install gpu4pyscf-cuda12x

# 3. Install CuPy and cuTENSOR matching CUDA 12 (crucial tensor contraction engines)
!pip install cupy-cuda12x cutensor-cu12

In [ ]:
!pip install -q pyscf-dispersion

import numpy as np
import pyscf
from pyscf import gto
from gpu4pyscf import dft                          # Fixed: Added missing dft import
from gpu4pyscf.solvent.grad import pcm as gpu_grad # Added for robust GPU solvent gradients
from pyscf.geomopt.geometric_solver import optimize

# 1. Mount Google Drive if not already done
from google.colab import drive
drive.mount('/content/drive')

# 2. Set the path to the XYZ file
file_path = "/content/drive/MyDrive/Axial_fluoro_start_EQ_OH.xyz"

# 3. Read the file, skipping the first 2 header lines (Atom Count & Comment)
with open(file_path, "r") as f:
    lines = f.readlines()[2:]
    cartesian_coordinates = "".join(lines)

print("Initialising molecular structure for charged pyran...")
mol = gto.Mole()
mol.atom = cartesian_coordinates
mol.basis = 'def2-tzvpd'  # Modern triple-zeta with diffuse and polarisation
mol.charge = 1            # Cationic complex
mol.spin = 0              # Closed-shell singlet
mol.build()

print("\nSetting up restricted DFT on GPU...")
mf = dft.RKS(mol).density_fit()
mf.xc = 'cam-b3lyp-d3bj'  # Activates CAM-B3LYP functional with Grimme's D3(BJ) dispersion

# Class method setup for implicit aqueous solvation
mf = mf.PCM()
mf.with_solvent.method = 'IEF-PCM'
mf.with_solvent.eps = 78.35         # Water dielectric
mf.with_solvent.lebedev_order = 29  # Dense grid mesh to avoid noise

# Explicitly bind the GPU PCM gradient constructor to ensure geomeTRIC
# has direct access to analytical solvent forces on the GPU.
mf.nuc_grad_method = lambda: gpu_grad.make_grad_object(mf)

print("\n====== LAUNCHING GPU-ACCELERATED GEOMETRY OPTIMIZATION ======")
# Run the structural relaxation via the standard geometric wrapper
mol_optimized = optimize(mf)

print("\n====== OPTIMISATION COMPLETE!")

# =====================================================================
# BLOCK 1: EXTRACT AND SAVE OPTIMISED GEOMETRY TO GOOGLE DRIVE
# =====================================================================

# 1. Get the final optimised geometry from the geomeTRIC output object
# This returns an internal PySCF molecule object with the new coordinates
mol_opt = mol_optimized

# 2. Extract elements and Cartesian coordinates (converted from Bohr back to Angstrom)
# PySCF stores coordinates internally in Bohr; converts them using the Bohr constant
from pyscf import __config__
import numpy as np

atoms_count = mol_opt.natm
coords_angstrom = mol_opt.atom_coords() * 0.529177210903  # Bohr to Angstrom conversion

# 3. Construct the XYZ file content
xyz_lines = []
xyz_lines.append(str(atoms_count))
xyz_lines.append(f"Optimized structure | Final Energy: {mf.e_tot:.8f} Hartree")

for i in range(atoms_count):
    symbol = mol_opt.atom_symbol(i)
    x, y, z = coords_angstrom[i]
    xyz_lines.append(f"{symbol:<3} {x:10.5f} {y:10.5f} {z:10.5f}")

# 4. Save directly to Google Drive
# Adjust the output filename as needed
opt_file_path = "/content/drive/MyDrive/Axial_fluoro_EQ_OH_OPT.xyz"

with open(opt_file_path, "w") as f:
    f.write("\n".join(xyz_lines))

print(f"🎉 Optimised structure successfully saved to: {opt_file_path}")


In [ ]:
# ========================================================================
# BLOCK 1: EXTRACT AND SAVE OPTIMISED GEOMETRY TO GOOGLE DRIVE WITH ENERGY
# ========================================================================

# 1. Get the final optimised geometry from the geomeTRIC output object
# This returns an internal PySCF molecule object with the new coordinates
mol_opt = mol_optimized

# NEW STEP: Calculate single-point energy at the final optimised geometry
final_mf = dft.RKS(mol_opt).density_fit()
final_mf.xc = 'cam-b3lyp-d3bj'
final_mf = final_mf.to_gpu()       # Use GPU acceleration
final_energy = final_mf.kernel()   # This computes the exact final energy

# 2. Extract elements and Cartesian coordinates
from pyscf import __config__
import numpy as np

atoms_count = mol_opt.natm
coords_angstrom = mol_opt.atom_coords() * 0.529177210903  # Bohr to Angstrom conversion

# 3. Construct the XYZ file content
xyz_lines = []
xyz_lines.append(str(atoms_count))
# FIX: Using the newly calculated final_energy variable
xyz_lines.append(f"Optimized structure | Final Energy: {final_energy:.8f} Hartree")

for i in range(atoms_count):
    symbol = mol_opt.atom_symbol(i)
    x, y, z = coords_angstrom[i]
    xyz_lines.append(f"{symbol:<3} {x:10.5f} {y:10.5f} {z:10.5f}")

# 4. Save directly to Google Drive
opt_file_path = "/content/drive/MyDrive/Axial_fluoro_EQ_OH_OPT_E.xyz"

with open(opt_file_path, "w") as f:
    f.write("\n".join(xyz_lines))

print(f"🎉 Optimised structure successfully saved to: {opt_file_path}")

In [ ]:
# =====================================================================
# CORRECTED BLOCK 2: HESSIAN / FREQUENCY CALCULATION (WITH PCM SOLVENT)
# =====================================================================
from pyscf import hessian

print("Starting Hessian calculation on the GPU... This may take a few minutes.")

# 1. Re-initialize and execute the DFT engine on your optimised structure (mol_opt)
mf_freq = dft.RKS(mol).density_fit()
mf_freq.xc = 'cam-b3lyp-d3bj'

# --- ACTIVATING IMPLICIT AQUEOUS SOLVATION FRAMEWORK ---
mf_freq = mf_freq.PCM()
mf_freq.with_solvent.method = 'IEF-PCM'
mf_freq.with_solvent.eps = 78.35         # Dielectric constant of water
mf_freq.with_solvent.lebedev_order = 29  # Dense integration grid mesh

# Port the complete solvated engine over to the GPU architecture
mf_freq = mf_freq.to_gpu()

# Run the single-point kernel first to converge the wave function
mf_freq.kernel()

# 2. Compute the Hessian matrix on the GPU (Now mf_freq is explicitly defined!)
hess_calc = mf_freq.Hessian()
hess_matrix = hess_calc.kernel()


# 3. Use PySCF's thermo module to convert the Hessian into vibrational frequencies
from pyscf.hessian import thermo
thermo_info = thermo.harmonic_analysis(mol, hess_matrix)

# Extract data using the proper lower-case/standard PySCF keys
frequencies = thermo_info.get('freq_wavenumber', [])

# Fallback mechanism for ZPE extraction depending on PySCF dictionary layout
zpve = thermo_info.get('zpe', thermo_info.get('ZPE', None))

# If neither key exists, calculate it manually from the real frequencies (0.5 * h * nu)
if zpve is None:
    # 1 Hartree = 219474.63 cm^-1, so conversion factor is 0.5 / 219474.63
    real_freqs = [f for f in frequencies if f > 0]
    zpve = sum(real_freqs) * 2.27817e-6

# 4. Print results and check for imaginary frequencies
print("\n" + "="*40)
print("VIBRATIONAL FREQUENCY RESULTS")
print("="*40)
print(f"Zero-Point Vibrational Energy (ZPE): {zpve:.6f} Hartree")
print(f"ZPE in kcal/mol: {zpve * 627.509:.3f} kcal/mol")

# Negative frequencies are returned as real numbers with a negative sign or complex formats
# We check for values significantly less than zero to identify imaginary modes
imaginary_freqs = [f for f in frequencies if f < -1.0]

print(f"\nTotal structural frequencies found: {len(frequencies)}")
if imaginary_freqs:
    print(f"⚠️ Warning: Found {len(imaginary_freqs)} imaginary frequency/frequencies!")
    print(f"Lowest imaginary modes: {imaginary_freqs}")
    print("This structure might be a transition state or a saddle point, not a true minimum.")
else:
    print("✅ Success! Zero imaginary frequencies found. This is a true local minimum structure.")


In [ ]:
# =====================================================================
# QUICK JUPYTER CELL: PRINT COMPLETED PYSCF VIBRATIONAL FREQUENCIES
# =====================================================================

print("=" * 60)
print("EXTRACTED VIBRATIONAL FREQUENCY SPECTRUM (cm^-1)")
print("=" * 60)

# Extract frequencies array from the active thermo_info dictionary object
frequencies_list = thermo_info.get('freq_wavenumber', [])

# Interrogate and print each mode from active session memory
for index, numeric_freq in enumerate(frequencies_list):
    if numeric_freq < -1.0:
        tag = "IMAGINARY ❌"
    elif -1.0 <= numeric_freq <= 30.0:
        tag = "TRIVIAL (Rotation/Translation)"
    else:
        tag = "ACTIVE VIBRATION"

    print(f"  Vibrational Mode {index+1:2d}: {numeric_freq:10.2f} cm^-1   [{tag}]")

print("=" * 60)
print(f"Total Structural Modes Extracted: {len(frequencies_list)}")


In [ ]:
# =====================================================================
# INDEPENDENT SPECTRUM EXPORTER
# =====================================================================

# 1. Path for the frequencies file
freq_only_path = "/content/drive/MyDrive/Axial_fluoro_EQ_OH_OPT_Raw.txt"

try:
    # 2. Extract and format into flat data rows
    raw_lines = []
    raw_lines.append("# Mode Number | Frequency (cm^-1)")

    for idx, freq in enumerate(frequencies, 1):
        raw_lines.append(f"{idx:<5} {freq:12.4f}")

    # 3. Write directly out to Drive
    with open(freq_only_path, "w") as f:
        f.write("\n".join(raw_lines))

    print(f"✅ Success! All {len(frequencies)} frequencies saved safely as a flat text file here:")
    print(freq_only_path)

except NameError:
    print("❌ Error: The 'frequencies' variable wasn't found in this cell's memory.")
    print("Please make sure you have run cell block successfully first!")


In [ ]:
# =====================================================================
# CORRECTED BLOCK 3: THERMODYNAMIC PROPERTIES
# =====================================================================
from pyscf.hessian import thermo

# 1. Run the thermochemical analysis using the mean-field object and frequencies
thermo_results = thermo.thermo(mf_freq, thermo_info['freq_wavenumber'])

# 2. Helper function to safely extract the numeric value from PySCF (Value, Unit) tuples
def get_val(item):
    if isinstance(item, tuple):
        return item[0]
    return item

print("\n" + "="*40)
print("THERMOCHEMICAL DATA AT 298.15 K, 1 ATM")
print("="*40)
print(f"Total Electronic Energy (E_scf) : {mf_freq.e_tot:.6f} Hartree")
print(f"Zero-Point Energy (ZPE)        : {zpve:.6f} Hartree")

# Extract the standard thermodynamic potentials (unwrapping the tuples)
e_tot = get_val(thermo_results['E_tot'])
h_tot = get_val(thermo_results['H_tot'])
g_tot = get_val(thermo_results['G_tot'])

print(f"Internal Energy (E_tot)        : {e_tot:.6f} au")
print(f"Enthalpy (H)                    : {h_tot:.6f} au")
print(f"Gibbs Free Energy (G)          : {g_tot:.6f} au")

# =====================================================================
# BLOCK 4: COMPILE DATA AND WRITE TO GOOGLE DRIVE REPORT
# =====================================================================

# Define your output path (change filename as desired)
report_path = "/content/drive/MyDrive/Axial_fluoro_EQ_OH_OPT_Report.txt"

# Build the text content structurally
report_content = f"""=====================================================================
QUANTUM CHEMISTRY COMPUTATION REPORT
=====================================================================

1. ELECTRONIC & GEOMETRY ENERGY
---------------------------------------------------------------------
Total Electronic Energy (E_scf) : {mf_freq.e_tot:.8f} Hartree

2. VIBRATIONAL FREQUENCY DATA
---------------------------------------------------------------------
Zero-Point Vibrational Energy (ZPE) : {zpve:.6f} Hartree
ZPE in kcal/mol                     : {zpve * 627.509:.3f} kcal/mol
Total Normal Modes Found            : {len(frequencies)}
Imaginary Frequencies Checked       : {"None (True Local Minimum)" if not [f for f in frequencies if f < -1.0] else "Warning: Imaginary modes present"}

3. THERMOCHEMICAL VALUES (298.15 K, 1 ATM)
---------------------------------------------------------------------
Internal Energy (E_tot)             : {e_tot:.8f} au
Enthalpy (H)                        : {h_tot:.8f} au
Gibbs Free Energy (G)               : {g_tot:.8f} au

=====================================================================
Report generated successfully via GPU-accelerated PySCF environment.
=====================================================================
"""

# Write the text to your drive
with open(report_path, "w") as report_file:
    report_file.write(report_content)

print(f"🎉 Complete summary log written! You can find it at: {report_path}")
print("You can download it directly from your Google Drive sidebar or interface.")

In [ ]:
import os
from pyscf.tools import molden

# =====================================================================
# JUPYTER CELL: DYNAMIC MOLDEN EXPORT & MOLECULAR FILE SAVE
# =====================================================================
filename = "charged_eq_pyran_solvated.molden"
print(f"Initiating full session data export to: {os.path.abspath(filename)}")
print("-" * 70)

# 1. Unpack data structures safely from active session memory
raw_frequencies = thermo_info.get('freq_wavenumber', [])
zpve_val = thermo_info.get('zpe', thermo_info.get('ZPE', None))

if zpve_val is None:
    real_modes = [f for f in raw_frequencies if f > 0]
    zpve_val = sum(real_modes) * 2.27817e-6

# 2. Extract and format the normal vibrational coordinates (displacement vectors)
# PySCF stores these inside the harmonic analysis dictionary
normal_modes = thermo_info.get('normal_modes', None)

# 3. Stream data into the Molden file structure
with open(filename, 'w') as f:
    # Header documentation block
    f.write("[Molden Format]\n")
    f.write(f"[Title]\n")
    f.write(f"  Protonated Fluoropyran Cation in Water\n")
    f.write(f"  Model Chemistry: CAM-B3LYP-D3BJ/def2-TZVPD/IEF-PCM\n")
    f.write(f"  Total Solvated Energy: {mf_freq.e_tot:.8f} Hartree\n")
    f.write(f"  Calculated ZPVE: {zpve_val:.6f} Hartree ({zpve_val * 627.509:.3f} kcal/mol)\n\n")

    # Export molecular geometry, basis set info, and orbital coefficients (Wavefunction)
    # Fixed: Stream directly to 'f' using granular molden functions instead of passing file object to from_scf
    molden.header(mol, f)
    molden.orbital_coeff(mol, f, mf_freq.mo_coeff, ene=mf_freq.mo_energy, occ=mf_freq.mo_occ)

    # 4. Append vibrational frequencies and normal modes if they exist in memory
    if len(raw_frequencies) > 0 and normal_modes is not None:
        f.write("[FREQ]\n")
        for freq in raw_frequencies:
            # Molden format traditionally expects real values; imaginary modes are written as negative
            f.write(f"  {freq:.4f}\n")

        f.write("\n[FR-COORD]\n")
        # Write out mass-weighted cartesian coordinates matching the normal modes
        coords_bohr = mol.atom_coords()
        for idx in range(mol.natm):
            sym = mol.atom_symbol(idx)
            pos = coords_bohr[idx]
            f.write(f" {sym:<2}   {pos[0]:14.8f} {pos[1]:14.8f} {pos[2]:14.8f}\n")

        f.write("\n[FR-NORM-COORD]\n")
        # Write down displacement directions for every atom per vibrational mode
        for mode_idx in range(len(raw_frequencies)):
            f.write(f"vibration {mode_idx + 1}\n")
            for atom_idx in range(mol.natm):
                # Pull the x, y, z displacement vectors for this specific atom and mode
                disp = normal_modes[mode_idx][atom_idx]
                f.write(f"  {disp[0]:14.8f} {disp[1]:14.8f} {disp[2]:14.8f}\n")

print(f"✅ EXPORT SUCCESSFUL: '{filename}' has been generated.")
print("   This file contains your complete quantum geometry, wave function matrices,")
print("   electronic energies, and interactive vibrational normal mode vectors.")
print("   You can open it directly in Avogadro, IboView, or Molden to visualize the orbitals.")